In [2]:
%load_ext sql
%sql sqlite:///advanced_sql.db

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


Connecting to 'sqlite:///advanced_sql.db'

In [3]:
%%sql
DROP TABLE IF EXISTS sales;
CREATE TABLE sales (
    sale_date DATE,
    salesperson TEXT,
    region TEXT,
    amount INTEGER
);

INSERT INTO sales (sale_date, salesperson, region, amount) VALUES 
('2026-02-01', 'Alice', 'North', 500),
('2026-02-02', 'Alice', 'North', 800),
('2026-02-03', 'Alice', 'North', 300),
('2026-02-01', 'Bob', 'South', 1000),
('2026-02-02', 'Bob', 'South', 200),
('2026-02-03', 'Bob', 'South', 900),
('2026-02-01', 'Charlie', 'North', 750),
('2026-02-02', 'Charlie', 'North', 600);

Running query in 'sqlite:///advanced_sql.db'

8 rows affected.

++
||
++
++

In [4]:
%%sql
SELECT 
    sale_date,
    salesperson,
    region,
    amount,
    SUM(amount) OVER(PARTITION BY region) AS total_regional_sales
FROM sales
ORDER BY region, sale_date;

Running query in 'sqlite:///advanced_sql.db'

sale_date,salesperson,region,amount,total_regional_sales
2026-02-01,Alice,North,500,2950
2026-02-01,Charlie,North,750,2950
2026-02-02,Alice,North,800,2950
2026-02-02,Charlie,North,600,2950
2026-02-03,Alice,North,300,2950
2026-02-01,Bob,South,1000,2100
2026-02-02,Bob,South,200,2100
2026-02-03,Bob,South,900,2100


In [6]:
%%sql
SELECT sale_date, salesperson, region, SUM(AMOUNT) OVER(PARTITION BY region) AS total_regional_sales
FROM sales
ORDER BY sale_date;

Running query in 'sqlite:///advanced_sql.db'

sale_date,salesperson,region,total_regional_sales
2026-02-01,Alice,North,2950
2026-02-01,Charlie,North,2950
2026-02-01,Bob,South,2100
2026-02-02,Alice,North,2950
2026-02-02,Charlie,North,2950
2026-02-02,Bob,South,2100
2026-02-03,Alice,North,2950
2026-02-03,Bob,South,2100


In [7]:
%%sql
SELECT 
    salesperson,
    region,
    amount,
    RANK() OVER(PARTITION BY region ORDER BY amount DESC) as regional_rank
FROM sales;

Running query in 'sqlite:///advanced_sql.db'

salesperson,region,amount,regional_rank
Alice,North,800,1
Charlie,North,750,2
Charlie,North,600,3
Alice,North,500,4
Alice,North,300,5
Bob,South,1000,1
Bob,South,900,2
Bob,South,200,3


In [8]:
%%sql
SELECT 
    sale_date,
    salesperson,
    amount,
    SUM(amount) OVER(
        PARTITION BY salesperson 
        ORDER BY sale_date
    ) AS running_total
FROM sales;

Running query in 'sqlite:///advanced_sql.db'

sale_date,salesperson,amount,running_total
2026-02-01,Alice,500,500
2026-02-02,Alice,800,1300
2026-02-03,Alice,300,1600
2026-02-01,Bob,1000,1000
2026-02-02,Bob,200,1200
2026-02-03,Bob,900,2100
2026-02-01,Charlie,750,750
2026-02-02,Charlie,600,1350


In [9]:
%%sql
SELECT 
    sale_date,
    salesperson,
    amount AS todays_sales,
    LAG(amount, 1) OVER(PARTITION BY salesperson ORDER BY sale_date) AS yesterdays_sales,
    amount - LAG(amount, 1) OVER(PARTITION BY salesperson ORDER BY sale_date) AS daily_difference
FROM sales;

Running query in 'sqlite:///advanced_sql.db'

sale_date,salesperson,todays_sales,yesterdays_sales,daily_difference
2026-02-01,Alice,500,None,None
2026-02-02,Alice,800,500,300
2026-02-03,Alice,300,800,-500
2026-02-01,Bob,1000,None,None
2026-02-02,Bob,200,1000,-800
2026-02-03,Bob,900,200,700
2026-02-01,Charlie,750,None,None
2026-02-02,Charlie,600,750,-150
